### Building ID mapping

In [30]:
from lexical_benchmark.datasets.childes import data2 as childes
from lexical_benchmark.utils import timed_status

dataset = childes.CHILDESDataset()
with timed_status(status="Scrapping for IDs", complete_status="IDs mapping build successfuly"):
    root_dir = dataset.source_path / "Eng-NA"
    id_list = [(item.relative_to(root_dir).parent / item.stem).parts for item in root_dir.rglob("*.cha")]
    (dataset.raw_path / "Eng-NA" / "ids.txt").write_text("\n".join([",".join(parts) for parts in id_list]))
    (dataset.clean_path / "Eng-NA" / "ids.txt").write_text("\n".join([",".join(parts) for parts in id_list]))

    root_dir = dataset.source_path / "Eng-UK"
    id_list = [(item.relative_to(root_dir).parent / item.stem).parts for item in root_dir.rglob("*.cha")]
    (dataset.raw_path / "Eng-UK" / "ids.txt").write_text("\n".join([",".join(parts) for parts in id_list]))
    (dataset.clean_path / "Eng-UK" / "ids.txt").write_text("\n".join([",".join(parts) for parts in id_list]))

Output()

IDs mapping build successfuly (Total time: 1 seconds)

### Compute Rejection/Acceptance Word Stats from dictionairy cleaning step

To be able to compare those stats with STELA and other datasets of different size we do a block-avergage computation.

In [1]:
from lexical_benchmark.datasets.childes import data2 as childes
from lexical_benchmark.utils import timed_status

dataset = childes.CHILDESDataset()

with timed_status(status="Gathering Words Eng-NA", complete_status="Finished Gathering Words from Eng-NA"):
    child_eng_na_words = []
    adult_eng_na_words = []
    for item in dataset.iter_accent("Eng-NA"):
        child_eng_na_words.extend(item.meta.child_source.read_tokenized())
        adult_eng_na_words.extend(item.meta.adult_source.read_tokenized())

%store child_eng_na_words
%store adult_eng_na_words

with timed_status(status="Gathering Words Eng-UK", complete_status="Finished Gathering Words from Eng-UK"):
    child_eng_uk_words = []
    adult_eng_uk_words = []
    for item in dataset.iter_accent("Eng-UK"):
        child_eng_uk_words.extend(item.meta.child_source.read_tokenized())
        adult_eng_uk_words.extend(item.meta.adult_source.read_tokenized())

%store child_eng_uk_words
%store adult_eng_uk_words

Output()

Finished Gathering Words from Eng-NA (Total time: 10 seconds)

Output()

Finished Gathering Words from Eng-UK (Total time: 6 seconds)

Stored 'child_eng_na_words' (list)
Stored 'adult_eng_na_words' (list)
Stored 'child_eng_uk_words' (list)
Stored 'adult_eng_uk_words' (list)


In [2]:
%store -r child_eng_na_words
%store -r adult_eng_na_words
%store -r child_eng_uk_words
%store -r adult_eng_uk_words
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average
from lexical_benchmark.datasets import utils

CHUNK_SIZE = 16_000
en_cleaner = utils.DictionairyCleaner(lang="EN", childes_extra_id="e9c0d55cf62ba9e937fe273de7626661")

with timed_status(status="Computing Rates for Eng_NA/child", complete_status="Finished Eng_NA/child !"):
    word_chunk_list = block_average.split_and_fill_chunks(child_eng_na_words, chunk_size=CHUNK_SIZE)
    child_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_NA/adult", complete_status="Finished Eng_NA/adult !"):
    word_chunk_list = block_average.split_and_fill_chunks(adult_eng_na_words, chunk_size=CHUNK_SIZE)
    adult_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_UK/child", complete_status="Finished Eng_UK/child !"):
    word_chunk_list = block_average.split_and_fill_chunks(child_eng_uk_words, chunk_size=CHUNK_SIZE)
    child_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )
with timed_status(status="Computing Rates for Eng_UK/adult", complete_status="Finished Eng_UK/adult !"):
    word_chunk_list = block_average.split_and_fill_chunks(adult_eng_uk_words, chunk_size=CHUNK_SIZE)
    adult_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

view_cfg = {"view_type": "result_tokens", "avg_type": "median"}
childes_word_stats_tables_16k_tokens = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
view_cfg = {"view_type": "result_types", "avg_type": "median"}
childes_word_stats_tables_16k_types = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
%store childes_word_stats_tables_16k_tokens
%store childes_word_stats_tables_16k_types

Output()

Finished Eng_NA/child ! (Total time: 2 seconds)

Output()

Finished Eng_NA/adult ! (Total time: 6 seconds)

Output()

Finished Eng_UK/child ! (Total time: 2 seconds)

Output()

Finished Eng_UK/adult ! (Total time: 6 seconds)

Stored 'childes_word_stats_tables_16k_tokens' (dict)
Stored 'childes_word_stats_tables_16k_types' (dict)


In [3]:
%store -r child_eng_na_words
%store -r adult_eng_na_words
%store -r child_eng_uk_words
%store -r adult_eng_uk_words
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average
from lexical_benchmark.datasets import utils

CHUNK_SIZE = 1600
en_cleaner = utils.DictionairyCleaner(lang="EN", childes_extra_id="e9c0d55cf62ba9e937fe273de7626661")

with timed_status(status="Computing Rates for Eng_NA/child", complete_status="Finished Eng_NA/child !"):
    word_chunk_list = block_average.split_and_fill_chunks(child_eng_na_words, chunk_size=CHUNK_SIZE)
    child_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_NA/adult", complete_status="Finished Eng_NA/adult !"):
    word_chunk_list = block_average.split_and_fill_chunks(adult_eng_na_words, chunk_size=CHUNK_SIZE)
    adult_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_UK/child", complete_status="Finished Eng_UK/child !"):
    word_chunk_list = block_average.split_and_fill_chunks(child_eng_uk_words, chunk_size=CHUNK_SIZE)
    child_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )
with timed_status(status="Computing Rates for Eng_UK/adult", complete_status="Finished Eng_UK/adult !"):
    word_chunk_list = block_average.split_and_fill_chunks(adult_eng_uk_words, chunk_size=CHUNK_SIZE)
    adult_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

view_cfg = {"view_type": "result_tokens", "avg_type": "median"}
childes_word_stats_tables_1k6h_tokens = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
view_cfg = {"view_type": "result_types", "avg_type": "median"}
childes_word_stats_tables_1k6h_types = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
%store childes_word_stats_tables_1k6h_tokens
%store childes_word_stats_tables_1k6h_types

Output()

Finished Eng_NA/child ! (Total time: 3 seconds)

Output()

Finished Eng_NA/adult ! (Total time: 6 seconds)

Output()

Finished Eng_UK/child ! (Total time: 3 seconds)

Output()

Finished Eng_UK/adult ! (Total time: 6 seconds)

Stored 'childes_word_stats_tables_1k6h_tokens' (dict)
Stored 'childes_word_stats_tables_1k6h_types' (dict)


In [6]:
%store -r childes_word_stats_tables_1k6h_tokens
%store -r childes_word_stats_tables_1k6h_types
%store -r childes_word_stats_tables_16k_tokens
%store -r childes_word_stats_tables_16k_types
import pandas as pd
from IPython.display import display_html

from lexical_benchmark.utils import ipython_utils

display_html("<h3> 16k block averages.</h3>")
ipython_utils.display_side_by_side(
    dataframes={
        "CHILDES 16k Block Average": (
            ("Tokens", pd.DataFrame(list(childes_word_stats_tables_16k_tokens.values()))),
            ("Types", pd.DataFrame(list(childes_word_stats_tables_16k_types.values())))
        ),
        "CHILDES 1.6k Block Average": (
            ("Tokens", pd.DataFrame(list(childes_word_stats_tables_1k6h_tokens.values()))),
            ("Types", pd.DataFrame(list(childes_word_stats_tables_1k6h_types.values())))
        ),
    },
    custom_format={
        'Tokens': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.2%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.2%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.2%}'
    },
)

,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
0,Eng-NA/child,"2,896,000","30,546",0.71%,"2,865,454",99.29%
1,Eng-NA/adult,"7,952,000","36,901",0.36%,"7,915,099",99.64%
2,Eng-UK/child,"2,960,000","11,465",0.33%,"2,948,535",99.67%
3,Eng-UK/adult,"8,032,000","20,554",0.24%,"8,011,446",99.76%
,Section,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,Eng-NA/child,"281,272","14,058",3.68%,"267,214",96.32%
1,Eng-NA/adult,"724,462","15,184",1.82%,"709,278",98.18%
2,Eng-UK/child,"257,394","5,388",1.69%,"252,006",98.31%
3,Eng-UK/adult,"734,257","9,461",1.16%,"724,796",98.84%
,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
